# DT401 Week 5 Section 3: Testing a Hypothesis

**Module:** Data Fundamentals  
**Week:** 5 of 7  
**Section:** 3 of 3  
**Estimated Time:** 50 minutes

---

## Learning Objective

Understand and apply statistical hypothesis testing to validate analytical findings.

By the end of this notebook, you will be able to:
- Formulate null and alternative hypotheses
- Understand statistical significance and p-values
- Test hypotheses about relationships in data
- Interpret hypothesis test results for business decisions

---

## Context Recap

In Section 2, you built regression and clustering models. But how do you know if the patterns you found are genuine or just chance? Hypothesis testing provides the statistical framework to answer this question rigorously. You'll learn to test whether observed relationships are statistically significant—essential for making evidence-based business decisions.

**Weather data from Open-Meteo API (https://open-meteo.com) - London, UK (51.49°N, 0.16°W), Jan 2023-Jan 2026**

---

## How to Use This Notebook

This notebook contains a mix of instructional text, Python code cells, and interactive exercises. Follow the instructions in each section, run the code cells to see outputs, and complete the exercises to reinforce your learning.

You will find several types of boxes throughout the notebook to guide your learning:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b> <br>
These boxes contain important reminders about how this section relates to the overall module and your learning journey.
</div>

<div class="alert alert-block alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the topic discussed.
</div>

<div class="alert alert-block alert-success">
<b>Task boxes:</b> <br>
These have questions or tasks you should complete before progressing.

These may include hidden hints or solutions.
<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Hints help guide your thinking without giving away the full answer.

</details>

</div>

---

## Part 1: Understanding Hypotheses

### Visual Overview

```mermaid
graph TD
    A[Research Question] --> B[Formulate Hypotheses]
    B --> C[Null Hypothesis H₀:<br/>No Effect/Difference]
    B --> D[Alternative Hypothesis H₁:<br/>Effect Exists]
    C --> E[Collect Data]
    D --> E
    E --> F[Calculate p-value]
    F --> G{p < 0.05?}
    G -->|Yes| H[Reject H₀<br/>Support H₁]
    G -->|No| I[Fail to Reject H₀<br/>Insufficient Evidence]
    
    style C fill:#ffe1e1
    style D fill:#e1ffe1
    style H fill:#e1ffe1
    style I fill:#ffe1e1
```

**What this diagram shows:**
Hypothesis testing follows a structured process: formulate null (no effect) and alternative (effect exists) hypotheses, collect data, calculate probability of results under null hypothesis (p-value), and decide whether to reject the null hypothesis based on significance threshold.

**Why this matters in your role:**
Hypothesis testing prevents you from acting on random noise. It provides statistical rigor to business decisions—ensuring you're responding to genuine patterns, not coincidences.

In [ ]:
# Load libraries and data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Load weather dataset
weather_data = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, low_memory=False)
weather_data.columns = [c.split(' ')[0].strip() for c in weather_data.columns]
weather_data = weather_data.rename(columns={'time': 'date'})

# Keep only hourly rows (YYYY-MM-DDTHH:MM format)
# The CSV contains a second daily-data section appended after the hourly data
weather_data = weather_data[weather_data['date'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)

# Convert all measurement columns to numeric
for col in weather_data.columns[1:]:
    weather_data[col] = pd.to_numeric(weather_data[col], errors='coerce')
weather_data['wind_speed_10m'] = weather_data['wind_speed_10m'] * 1.60934

print(f"Dataset loaded: {len(weather_data):,} observations")
weather_data.head()

---

### What Is a Hypothesis?

A hypothesis is a precise, testable statement about expected outcomes. In data analysis, hypotheses are always stated in pairs:

**Null Hypothesis (H₀):**
- Your baseline assumption
- Often "no effect" or "no difference"
- What you assume is true unless data proves otherwise
- Example: "Temperature has no effect on wind speed"

**Alternative Hypothesis (H₁ or Hₐ):**
- Your proposed theory
- What you hope to demonstrate with data
- Example: "Higher temperatures increase wind speed"

**Business Context:**
- **Healthcare:** H₀: New treatment has no effect; H₁: New treatment reduces symptoms
- **Retail:** H₀: Promotion has no impact on sales; H₁: Promotion increases sales
- **Manufacturing:** H₀: Process change doesn't affect quality; H₁: Process change improves quality

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Science proceeds by falsification—we can disprove theories but never absolutely prove them. We assume the null hypothesis until data provides sufficient evidence to reject it. This conservative approach guards against seeing patterns in random noise.
</div>

---

### Worked Example: Testing Temperature-Wind Relationship

**Research Question:** Does temperature affect wind speed?

**Hypotheses:**
- H₀: Temperature has no effect on wind speed (correlation = 0)
- H₁: Temperature affects wind speed (correlation ≠ 0)

In [ ]:
# Prepare data for hypothesis testing
test_data = weather_data[['temperature_2m', 'wind_speed_10m']].dropna()

# Visualize relationship
plt.figure(figsize=(10, 6))
sample = test_data.sample(n=500, random_state=42)  # Sample for clarity
plt.scatter(sample['temperature_2m'], sample['wind_speed_10m'], alpha=0.5)
plt.xlabel('Temperature (°C)')
plt.ylabel('Wind Speed (km/h)')
plt.title('Temperature vs Wind Speed')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Visual inspection suggests a relationship may exist")
print("But is it statistically significant or just chance?")

In [ ]:
# Calculate correlation and test significance
correlation, p_value = stats.pearsonr(test_data['temperature_2m'], test_data['wind_speed_10m'])

print("=== Hypothesis Test Results ===")
print(f"\nCorrelation coefficient: {correlation:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"\nInterpretation:")
print(f"- Correlation strength: {'Weak' if abs(correlation) < 0.3 else 'Moderate' if abs(correlation) < 0.7 else 'Strong'}")
print(f"- Direction: {'Positive' if correlation > 0 else 'Negative'}")

if p_value < 0.05:
    print(f"\n✓ REJECT NULL HYPOTHESIS (p < 0.05)")
    print(f"Conclusion: Temperature significantly affects wind speed")
else:
    print(f"\n✗ FAIL TO REJECT NULL HYPOTHESIS (p ≥ 0.05)")
    print(f"Conclusion: Insufficient evidence that temperature affects wind speed")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We calculated Pearson correlation coefficient (measures linear relationship strength)
- We obtained p-value (probability of seeing this correlation if H₀ were true)
- p-value < 0.05 means: "If temperature truly had no effect, we'd see a correlation this strong in less than 5% of samples"
- Therefore, we reject H₀ and conclude temperature does affect wind speed

**Key lesson:** P-values quantify evidence against the null hypothesis. Lower p-value = stronger evidence for alternative hypothesis.

</div>

---

## Part 2: Understanding Statistical Significance

### Visual Overview

```mermaid
graph LR
    A[P-value] --> B{Compare to α}
    B -->|p < 0.05| C[Statistically Significant<br/>Reject H₀]
    B -->|p ≥ 0.05| D[Not Significant<br/>Fail to Reject H₀]
    
    C --> E[Effect likely genuine]
    D --> F[Could be chance]
    
    style C fill:#e1ffe1
    style D fill:#ffe1e1
```

**What this diagram shows:**
Statistical significance compares your p-value against a threshold (α, typically 0.05). Results are significant if p < α, meaning they're unlikely to occur by chance.

**Why this matters in your role:**
Significance thresholds prevent you from overreacting to random variation. A significant result gives you confidence to act; a non-significant result suggests waiting for more evidence.

---

### Interpreting P-Values

**P-value meaning:** "If the null hypothesis were true, what's the probability of seeing results this extreme?"

**Common interpretations:**
- **p < 0.001:** Very strong evidence against H₀ ("highly significant")
- **p < 0.01:** Strong evidence against H₀
- **p < 0.05:** Moderate evidence against H₀ ("statistically significant")
- **p ≥ 0.05:** Insufficient evidence against H₀

**Business Context - Retail Example:**
Testing whether new website design increases sales:
- **p = 0.001:** Very strong evidence new design works → implement immediately
- **p = 0.03:** Moderate evidence → implement with monitoring
- **p = 0.15:** Insufficient evidence → test longer or abandon change

### Worked Example: Testing Multiple Relationships

Let's test several weather variable relationships to see which are statistically significant.

In [ ]:
# Test multiple relationships
test_pairs = [
    ('temperature_2m', 'relative_humidity_2m', 'Temperature vs Humidity'),
    ('temperature_2m', 'precipitation', 'Temperature vs Precipitation'),
    ('wind_speed_10m', 'precipitation', 'Wind Speed vs Precipitation'),
    ('relative_humidity_2m', 'precipitation', 'Humidity vs Precipitation')
]

print("=== Testing Multiple Hypotheses ===")
print(f"\nSignificance threshold (α): 0.05\n")

results = []
for var1, var2, label in test_pairs:
    data = weather_data[[var1, var2]].dropna()
    corr, p = stats.pearsonr(data[var1], data[var2])
    
    sig = "✓ SIGNIFICANT" if p < 0.05 else "✗ NOT SIGNIFICANT"
    results.append({'Relationship': label, 'Correlation': corr, 'P-value': p, 'Significant': sig})
    
    print(f"{label:35} | r={corr:7.4f} | p={p:.6f} | {sig}")

# Convert to DataFrame for summary
results_df = pd.DataFrame(results)
print(f"\nSummary: {(results_df['P-value'] < 0.05).sum()} of {len(results_df)} relationships are statistically significant")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We tested four different variable pairs for correlation
- Each test produced a correlation coefficient and p-value
- We compared p-values to α = 0.05 threshold
- Significant relationships (p < 0.05) indicate genuine effects worth investigating

**Key lesson:** Not all observed patterns are statistically significant. Hypothesis testing separates genuine relationships from random noise.

</div>

---

### Modified Task: Test Your Own Hypotheses

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Choose two weather variables not tested above. Formulate H₀ and H₁, test the relationship, and interpret results.

Available variables: temperature_2m, relative_humidity_2m, dew_point_2m, precipitation, rain, wind_speed_10m, wind_direction_10m, surface_pressure, cloud_cover

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Structure:
1. State your hypotheses explicitly
   - H₀: Variable X has no relationship with variable Y
   - H₁: Variable X is related to variable Y
2. Extract variables and remove missing values
3. Calculate correlation: stats.pearsonr(x, y)
4. Compare p-value to 0.05
5. State conclusion in business terms

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Example: Test dew point vs cloud cover
print("Hypothesis Test: Dew Point vs Cloud Cover\n")
print("H₀: Dew point has no relationship with cloud cover")
print("H₁: Dew point is related to cloud cover\n")

# Prepare data
test_data = weather_data[['dew_point_2m', 'cloud_cover']].dropna()

# Calculate correlation and p-value
corr, p = stats.pearsonr(test_data['dew_point_2m'], test_data['cloud_cover'])

print(f"Correlation: {corr:.4f}")
print(f"P-value: {p:.6f}")

if p < 0.05:
    print(f"\n✓ Reject H₀: Significant relationship found (p < 0.05)")
    print(f"Conclusion: Dew point significantly relates to cloud cover")
else:
    print(f"\n✗ Fail to reject H₀: No significant relationship (p ≥ 0.05)")
    print(f"Conclusion: Insufficient evidence of dew point-cloud cover relationship")
```

</details>

</div>

In [ ]:
# Your hypothesis test:
# Choose two variables and test their relationship

# 1. State your hypotheses
print("My Hypothesis Test:")
print("H₀: ")
print("H₁: ")

# 2. Prepare data (replace 'var1' and 'var2' with your chosen variables)
# test_data = weather_data[['var1', 'var2']].dropna()

# 3. Calculate correlation and p-value


# 4. Interpret results


print("\nResults:")

**Expected output:** Correlation coefficient, p-value, and conclusion stating whether to reject H₀

**Success check:** You can explain what the p-value means for your specific hypothesis and whether the relationship is statistically significant

---

## Part 3: Hypothesis Testing in Model Building

### Visual Overview

```mermaid
graph TD
    A[Build Regression Model] --> B[Model Coefficients]
    B --> C[Test Each Coefficient]
    C --> D{p < 0.05?}
    D -->|Yes| E[Variable Significantly<br/>Contributes to Model]
    D -->|No| F[Variable May Not<br/>Add Value]
    E --> G[Keep in Model]
    F --> H[Consider Removing]
    
    style E fill:#e1ffe1
    style F fill:#ffe1e1
```

**What this diagram shows:**
When building regression models, you're testing hypotheses about each variable's coefficient. Significant coefficients (p < 0.05) indicate variables that genuinely contribute to predictions.

**Why this matters in your role:**
This helps you build parsimonious models—including only variables that add real predictive value. Simpler models are easier to explain to stakeholders and less likely to overfit.

---

### Worked Example: Testing Model Coefficients

Let's build a regression model and test whether each variable's coefficient is significantly different from zero.

In [ ]:
# Build regression model with multiple predictors
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

# Predict wind speed from temperature, humidity, and pressure
model_data = weather_data[['temperature_2m', 'relative_humidity_2m', 'surface_pressure', 'wind_speed_10m']].dropna()

X = model_data[['temperature_2m', 'relative_humidity_2m', 'surface_pressure']]
y = model_data['wind_speed_10m']

# Add constant (intercept) for statsmodels
X_with_const = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X_with_const).fit()

print("=== Regression Model Summary ===")
print(model.summary())

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The model output shows hypothesis tests for each coefficient:
- **coef:** Coefficient value (effect size)
- **P>|t|:** P-value testing H₀: coefficient = 0
- **[0.025, 0.975]:** 95% confidence interval for coefficient

For each variable, we're testing:
- H₀: This variable has no effect (coefficient = 0)
- H₁: This variable affects wind speed (coefficient ≠ 0)

**Key lesson:** Variables with p < 0.05 significantly contribute to predictions. Variables with p ≥ 0.05 may not add value—consider removing them.

</div>

Let's interpret the coefficient tests more clearly:

In [ ]:
# Extract and interpret coefficient p-values
print("=== Coefficient Significance Tests ===")
print(f"\nSignificance threshold (α): 0.05\n")

for var, p_val in zip(['Intercept'] + list(X.columns), model.pvalues):
    sig = "✓ SIGNIFICANT" if p_val < 0.05 else "✗ NOT SIGNIFICANT"
    print(f"{var:25} | p = {p_val:.6f} | {sig}")

print("\n=== Business Interpretation ===")
sig_vars = [var for var, p in zip(list(X.columns), model.pvalues[1:]) if p < 0.05]
print(f"Significant predictors: {', '.join(sig_vars) if sig_vars else 'None'}")
print(f"\nThese variables genuinely contribute to wind speed predictions.")
print(f"Non-significant variables may be redundant or not truly related to wind speed.")

---

## Part 4: Independent Task - Complete Hypothesis Testing Analysis

### Your Challenge

You work for a charity evaluating whether donor age affects donation frequency. You have historical donor data with:
- Donor age (years)
- Donations per year (frequency)
- Average donation amount (£)
- Years as supporter
- Engagement score (event attendance, email opens)

**Conduct a complete hypothesis testing analysis:**

1. **Primary Question:** Does donor age significantly relate to donation frequency?
   - State H₀ and H₁
   - Determine what test would be appropriate
   - Describe how you'd interpret p-value results
   - Explain business implications for significant vs non-significant results

2. **Secondary Questions:** Identify 3 other relationships worth testing
   - For each, state H₀ and H₁
   - Explain why testing this relationship matters for fundraising strategy
   - Describe actions you'd take based on test results

3. **Model Building:** Design a regression model predicting donation amount
   - List potential predictor variables
   - For each predictor, state the hypothesis being tested
   - Describe how coefficient p-values would guide model refinement
   - Explain how you'd communicate significant findings to fundraising team

**Getting started:**
- Think about donor behavior patterns you've observed
- Consider what relationships would be actionable
- Remember: significance doesn't always mean causation—correlation might reflect confounding factors

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Complete the hypothesis testing analysis template for the charity donor scenario.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

**Primary Question Structure:**
- H₀: Age has no relationship with donation frequency (correlation = 0)
- H₁: Age relates to donation frequency (correlation ≠ 0)
- Test: Pearson correlation
- If p < 0.05: Age significantly predicts frequency → tailor appeals by age group
- If p ≥ 0.05: Age doesn't predict frequency → focus on other factors

**Secondary Questions Examples:**
- Years supporting vs donation amount
- Engagement score vs donation frequency
- Age vs preferred communication channel

**Model Building:**
- Predictors: Age, years supporting, engagement score, frequency
- Each coefficient tests: Does this variable significantly predict donation amount?
- Significant predictors (p < 0.05) = focus areas for fundraising optimization

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
analysis = {
    'primary_question': {
        'question': 'Does donor age significantly relate to donation frequency?',
        'H0': 'Age has no relationship with donation frequency (correlation = 0)',
        'H1': 'Age relates to donation frequency (correlation ≠ 0)',
        'test_type': 'Pearson correlation coefficient',
        'interpretation_significant': 'If p < 0.05: Reject H₀. Age significantly predicts frequency. Action: Segment donors by age group; tailor appeal timing and messaging to age-specific preferences.',
        'interpretation_not_significant': 'If p ≥ 0.05: Fail to reject H₀. Age doesn\'t significantly predict frequency. Action: Focus fundraising strategy on engagement score and giving history rather than demographics.'
    },
    'secondary_questions': [
        {
            'relationship': 'Years as supporter vs Average donation amount',
            'H0': 'Length of support has no relationship with donation size',
            'H1': 'Longer supporters give larger donations',
            'business_importance': 'If significant, prioritize donor retention programs to cultivate larger gifts over time',
            'actions': 'Significant: Invest in retention. Not significant: Focus acquisition equally with retention.'
        },
        {
            'relationship': 'Engagement score vs Donation frequency',
            'H0': 'Engagement level doesn\'t affect giving frequency',
            'H1': 'Higher engagement increases donation frequency',
            'business_importance': 'If significant, engagement programs directly impact revenue',
            'actions': 'Significant: Expand engagement activities (events, updates). Not significant: Engagement is relationship-building, not revenue driver.'
        },
        {
            'relationship': 'Donation frequency vs Average donation amount',
            'H0': 'Frequency and amount are independent',
            'H1': 'Frequency relates to donation size',
            'business_importance': 'Determines whether to encourage more frequent giving or larger individual gifts',
            'actions': 'Positive correlation: Frequent givers also give more—encourage monthly giving. Negative: Choice between frequency and size—segment strategies.'
        }
    ],
    'model_building': {
        'dependent_variable': 'Average donation amount (£)',
        'predictors': [
            {'variable': 'Age', 'hypothesis_H0': 'Age coefficient = 0 (no age effect)', 'hypothesis_H1': 'Age coefficient ≠ 0'},
            {'variable': 'Years supporting', 'hypothesis_H0': 'Years coefficient = 0', 'hypothesis_H1': 'Years coefficient ≠ 0'},
            {'variable': 'Donation frequency', 'hypothesis_H0': 'Frequency coefficient = 0', 'hypothesis_H1': 'Frequency coefficient ≠ 0'},
            {'variable': 'Engagement score', 'hypothesis_H0': 'Engagement coefficient = 0', 'hypothesis_H1': 'Engagement coefficient ≠ 0'}
        ],
        'model_refinement': 'If predictor p-value ≥ 0.05, consider removing from model. Significant predictors (p < 0.05) are retained as they genuinely improve donation amount predictions.',
        'stakeholder_communication': 'Present significant predictors with confidence intervals: "We\'re 95% confident that each additional year of support increases donation size by £X-£Y. This justifies investing £Z in retention programs."'
    }
}
```

</details>

</div>

In [ ]:
# Your hypothesis testing analysis (write in code cell for portfolio)
analysis = {
    'primary_question': {
        'question': '',
        'H0': '',
        'H1': '',
        'test_type': '',
        'interpretation_significant': '',
        'interpretation_not_significant': ''
    },
    'secondary_questions': [
        {
            'relationship': '',
            'H0': '',
            'H1': '',
            'business_importance': '',
            'actions': ''
        },
        # Add 2 more secondary questions
    ],
    'model_building': {
        'dependent_variable': '',
        'predictors': [],  # List of dicts with variable, H0, H1
        'model_refinement': '',
        'stakeholder_communication': ''
    }
}

# Print your analysis
print("=== HYPOTHESIS TESTING ANALYSIS ===")
print(f"\nPrimary Question: {analysis['primary_question']['question']}")
print(f"H₀: {analysis['primary_question']['H0']}")
print(f"H₁: {analysis['primary_question']['H1']}")
print(f"\nTest: {analysis['primary_question']['test_type']}")
print(f"\nIf significant: {analysis['primary_question']['interpretation_significant']}")
print(f"\nIf not significant: {analysis['primary_question']['interpretation_not_significant']}")

print("\n=== SECONDARY QUESTIONS ===")
for i, q in enumerate(analysis['secondary_questions'], 1):
    print(f"\n{i}. {q['relationship']}")
    print(f"   H₀: {q['H0']}")
    print(f"   H₁: {q['H1']}")
    print(f"   Why test: {q['business_importance']}")
    print(f"   Actions: {q['actions']}")

print("\n=== MODEL BUILDING ===")
print(f"Predicting: {analysis['model_building']['dependent_variable']}")
print(f"\nPredictors and hypotheses:")
for p in analysis['model_building']['predictors']:
    if p:  # Only print if predictor added
        print(f"  - {p}")
print(f"\nRefinement approach: {analysis['model_building']['model_refinement']}")
print(f"\nCommunication: {analysis['model_building']['stakeholder_communication']}")

**Success criteria:**
- Clear H₀ and H₁ stated for all questions
- Business implications explained for both significant and non-significant results
- 3 secondary questions identify actionable relationships
- Model building approach uses coefficient significance for refinement
- Stakeholder communication translates statistical findings into business language

**Portfolio note:** This hypothesis testing analysis demonstrates your ability to apply statistical methods to business problems (MLO 3). Include this in your Module Challenge Part A portfolio.

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 150 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against **MLO3:** *Select and apply appropriate data-gathering methods for a given data analysis scenario*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. A statistically significant result doesn't always mean a practically important one. Describe a scenario where the result is statistically significant but the effect size is too small to matter for business, and one where non-significance might still warrant action.

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 5 Section 3.** You can now:

- Formulated null and alternative hypotheses and tested the temperature-wind speed relationship using `scipy.stats.pearsonr`, interpreting the resulting correlation coefficient and p-value
- Tested four weather variable pairs for statistical significance at alpha=0.05, then formulated and tested your own hypothesis on a pair of your choice
- Built a multiple regression model with `statsmodels.api.OLS` predicting wind speed from temperature, humidity, and pressure, and used each coefficient's p-value to judge whether that variable significantly contributes to the model
- Designed a complete hypothesis-testing analysis, primary question, secondary questions, and a regression model, for a charity's donor-behaviour dataset

**Key takeaways:**

- A p-value answers "if the null hypothesis were true, how likely is a result this extreme?" It is not the probability that the null hypothesis itself is true
- Statistical significance is not the same as practical importance: a small but genuine effect can produce p < 0.05, and the significance threshold itself is a judgement call, stricter for medical trials (alpha = 0.01) than for marketing tests (alpha = 0.10)

**What comes next:**

- **Week 6** turns from statistical rigour to professional and legal rigour, covering the standards, compliance obligations, and data anonymisation techniques, including GDPR/UK GDPR, the Data Protection Act 2018, and data ethics, that govern how the data you have been analysing must be handled.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 5 Section 3 Testing a Hypothesis"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.5 component on Aptem